# Phase 2A: Regression (Predicting Numbers)

## 🎯 Learning Objectives

By the end of this notebook, you will:

- ✅ Master regression algorithms for predicting numbers
- ✅ Build Linear, Multiple, and Polynomial Regression models
- ✅ Apply Regularization techniques (Ridge, Lasso, ElasticNet)
- ✅ Understand evaluation metrics for regression
- ✅ Build end-to-end prediction models

**Time Required:** 1-2 weeks  
**Difficulty:** Intermediate  
**Prerequisites:** Phases 0-1 completed

## 📊 What is Regression?

**Goal:** Predict a continuous numerical value.

**Examples:**
- 🏠 House prices ($200,000)
- 🌡️ Temperature (25.5°C)
- 📈 Stock prices ($150.75)
- 🚗 Car fuel efficiency (32.5 mpg)
- 💰 Salary ($75,000)

In [5]:
# Import all required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


---

## 2.1 Simple Linear Regression

### Concept
Find the best-fit straight line through data points.

**Formula:** $y = mx + b$

- $m$ = slope (how much y changes per unit of x)
- $b$ = y-intercept (value when x=0)
- $x$ = input feature
- $y$ = prediction

In [6]:
# Simple Example: Salary Prediction based on Experience

# Data: Years of experience vs Salary
experience = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]).reshape(-1, 1)
salary = np.array([30000, 35000, 42000, 48000, 55000, 60000, 67000, 75000, 82000, 90000])

# Create and train model
model = LinearRegression()
model.fit(experience, salary)

# Make predictions
predictions = model.predict(experience)

# Model parameters
print("=== Linear Regression Model ===")
print(f"Slope (m): ${model.coef_[0]:,.2f} per year")
print(f"Intercept (b): ${model.intercept_:,.2f}")
print(f"\nInterpretation: For each additional year of experience,")
print(f"               salary increases by ${model.coef_[0]:,.2f}")

# Evaluation
r2 = r2_score(salary, predictions)
mae = mean_absolute_error(salary, predictions)
print(f"\n=== Model Performance ===")
print(f"R² Score: {r2:.4f} ({r2*100:.1f}% variance explained)")
print(f"Mean Absolute Error: ${mae:,.2f}")

=== Linear Regression Model ===
Slope (m): $6,642.42 per year
Intercept (b): $21,866.67

Interpretation: For each additional year of experience,
               salary increases by $6,642.42

=== Model Performance ===
R² Score: 0.9972 (99.7% variance explained)
Mean Absolute Error: $751.52


In [7]:
# Predict for new value
new_experience = np.array([[12], [15], [20]])
predicted_salary = model.predict(new_experience)

print("=== Predictions for New Values ===")
for exp, sal in zip(new_experience, predicted_salary):
    print(f"  {exp[0]} years experience: ${sal:,.2f}")

=== Predictions for New Values ===
  12 years experience: $101,575.76
  15 years experience: $121,503.03
  20 years experience: $154,715.15


In [ ]:
# Visualize Linear Regression

plt.figure(figsize=(12, 6))

# Plot actual data
plt.scatter(experience, salary, color='blue', s=100, alpha=0.6, label='Actual data', zorder=5)

# Plot regression line
plt.plot(experience, predictions, color='red', linewidth=2, label='Linear regression line')

# Plot new predictions
plt.scatter(new_experience, predicted_salary, color='green', s=150, marker='*',
            label='New predictions', zorder=5, edgecolors='black', linewidths=1)

# Formatting
plt.xlabel('Years of Experience', fontsize=12)
plt.ylabel('Salary ($)', fontsize=12)
plt.title('Salary Prediction using Linear Regression', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

# Add equation on plot
equation = f'y = {model.coef_[0]:,.0f}x + {model.intercept_:,.0f}'
plt.text(2, 80000, f'Equation: {equation}', fontsize=11, 
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

### Understanding the Math (Simplified)

In [ ]:
# What the model does internally:
# 1. Try random line: y = m*x + b
# 2. Calculate error (how far off predictions are)
# 3. Adjust m and b to reduce error
# 4. Repeat until error is minimized

# Cost Function: Mean Squared Error (what we minimize)
def cost_function(y_true, y_pred):
    """Mean Squared Error - measures average squared difference"""
    return np.mean((y_true - y_pred) ** 2)

# Example
y_true = np.array([100, 200, 300])
y_pred_bad = np.array([150, 150, 150])    # All same prediction (bad)
y_pred_good = np.array([110, 190, 310])   # Close to actual (good)

print("Cost Function Comparison:")
print(f"Bad predictions (all same):   MSE = {cost_function(y_true, y_pred_bad):,.2f}")
print(f"Good predictions (close):     MSE = {cost_function(y_true, y_pred_good):,.2f}")
print(f"\n→ Lower MSE = Better model!")

---

## 2.2 Multiple Linear Regression

### Concept
Use **multiple features** to make predictions.

**Formula:** $y = b_0 + b_1x_1 + b_2x_2 + b_3x_3 + ...$

**Example:** House price depends on size, bedrooms, age, location, etc.

In [ ]:
# Multiple Linear Regression: House Price Prediction

# Create dataset
data = {
    'Size': [1400, 1600, 1700, 1875, 1100, 1550, 2350, 2450, 1425, 1700,
             1800, 2000, 2200, 1650, 1900, 2100, 2300, 1750, 2050, 1950],
    'Bedrooms': [3, 3, 2, 4, 2, 3, 4, 4, 3, 3, 3, 4, 4, 3, 3, 4, 5, 3, 4, 3],
    'Age': [0, 10, 15, 2, 20, 8, 5, 3, 12, 7, 6, 4, 1, 9, 11, 5, 2, 8, 3, 6],
    'Price': [245000, 312000, 279000, 308000, 199000, 219000, 405000, 424000,
              319000, 255000, 285000, 330000, 395000, 275000, 295000, 365000,
              440000, 290000, 355000, 320000]
}
df = pd.DataFrame(data)

print("=== House Price Dataset ===")
print(df.head(10))
print(f"\nDataset shape: {df.shape}")
print(f"\nFeatures: Size (sq ft), Bedrooms, Age (years)")
print(f"Target: Price ($)")

In [ ]:
# Explore data correlations
plt.figure(figsize=(10, 8))

# Correlation heatmap
correlation = df.corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            square=True, linewidths=1)
plt.title('Feature Correlations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCorrelation with Price:")
print(correlation['Price'].sort_values(ascending=False))

In [ ]:
# Train Multiple Linear Regression

# Features and target
X = df[['Size', 'Bedrooms', 'Age']]
y = df['Price']

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluation
print("\n=== Model Performance ===")
print(f"R² Score: {r2_score(y_test, y_pred):.3f}")
print(f"Mean Absolute Error: ${mean_absolute_error(y_test, y_pred):,.2f}")
print(f"Root Mean Squared Error: ${np.sqrt(mean_squared_error(y_test, y_pred)):,.2f}")

# Coefficients
print("\n=== Model Coefficients ===")
for feature, coef in zip(X.columns, model.coef_):
    print(f"  {feature}: ${coef:,.2f}")
print(f"  Intercept: ${model.intercept_:,.2f}")

# Interpretation
print("\n=== Interpretation ===")
print(f"  • Each additional sq ft increases price by ${model.coef_[0]:,.2f}")
print(f"  • Each additional bedroom increases price by ${model.coef_[1]:,.2f}")
print(f"  • Each year older decreases price by ${-model.coef_[2]:,.2f}")

In [ ]:
# Predict new house prices
new_houses = pd.DataFrame({
    'Size': [2000, 1500, 2500],
    'Bedrooms': [3, 2, 4],
    'Age': [5, 15, 0]
})

predictions = model.predict(new_houses)

print("=== New House Predictions ===")
for i, (_, house) in enumerate(new_houses.iterrows()):
    print(f"\nHouse {i+1}: {house['Size']} sq ft, {house['Bedrooms']} beds, {house['Age']} years old")
    print(f"  Predicted Price: ${predictions[i]:,.2f}")

In [ ]:
# Visualize Actual vs Predicted

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Actual vs Predicted
ax1 = axes[0]
ax1.scatter(y_test, y_pred, s=100, alpha=0.6)
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
         'r--', lw=2, label='Perfect prediction')
ax1.set_xlabel('Actual Price ($)', fontsize=12)
ax1.set_ylabel('Predicted Price ($)', fontsize=12)
ax1.set_title('Actual vs Predicted Prices', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Feature Importance
ax2 = axes[1]
features = X.columns
coefficients = model.coef_
colors = ['green' if c > 0 else 'red' for c in coefficients]
ax2.barh(features, coefficients, color=colors)
ax2.set_xlabel('Coefficient Value ($)', fontsize=12)
ax2.set_title('Feature Impact on Price', fontsize=13, fontweight='bold')
ax2.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 2.3 Polynomial Regression

### Concept
When data has a **curved relationship**, use polynomial terms.

**Formula:** $y = b_0 + b_1x + b_2x^2 + b_3x^3 + ...$

**When to use:** Data doesn't follow a straight line!

In [ ]:
# Polynomial Regression Example

# Generate non-linear data
np.random.seed(42)
X = np.linspace(0, 10, 50).reshape(-1, 1)
y = 2 + 3*X.ravel() - 0.5*X.ravel()**2 + np.random.normal(0, 2, 50)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Compare Linear vs Polynomial
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

degrees = [1, 2, 5]
titles = ['Linear (degree=1)', 'Polynomial (degree=2)', 'Polynomial (degree=5)']

for ax, degree, title in zip(axes, degrees, titles):
    # Create polynomial model
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X_train, y_train)
    
    # Predictions
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Calculate R² scores
    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)
    
    # Plot
    X_plot = np.linspace(0, 10, 200).reshape(-1, 1)
    y_plot = model.predict(X_plot)
    
    ax.scatter(X_train, y_train, color='blue', alpha=0.6, label='Train', s=50)
    ax.scatter(X_test, y_test, color='orange', alpha=0.6, label='Test', s=50)
    ax.plot(X_plot, y_plot, 'r-', linewidth=2, label='Model')
    ax.set_xlabel('X')
    ax.set_ylabel('y')
    ax.set_title(f'{title}\nTrain R²: {r2_train:.3f} | Test R²: {r2_test:.3f}', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observations:")
print("- Degree 1: Underfitting (can't capture curve)")
print("- Degree 2: Good fit (captures the pattern)")
print("- Degree 5: Potential overfitting (too wiggly)")

In [ ]:
# Find Optimal Polynomial Degree

degrees = range(1, 15)
train_scores = []
test_scores = []

for degree in degrees:
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X_train, y_train)
    
    train_scores.append(r2_score(y_train, model.predict(X_train)))
    test_scores.append(r2_score(y_test, model.predict(X_test)))

# Plot
plt.figure(figsize=(12, 6))
plt.plot(degrees, train_scores, 'bo-', label='Train R²', linewidth=2, markersize=8)
plt.plot(degrees, test_scores, 'ro-', label='Test R²', linewidth=2, markersize=8)
plt.xlabel('Polynomial Degree', fontsize=12)
plt.ylabel('R² Score', fontsize=12)
plt.title('Choosing Optimal Polynomial Degree', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.axvline(x=2, color='green', linestyle='--', label='Optimal (degree=2)')

# Mark optimal
best_degree = degrees[np.argmax(test_scores)]
plt.annotate(f'Best degree={best_degree}', xy=(best_degree, max(test_scores)),
             xytext=(best_degree+2, max(test_scores)-0.1),
             arrowprops=dict(arrowstyle='->', color='green'),
             fontsize=11, color='green')

plt.tight_layout()
plt.show()

print(f"Best polynomial degree: {best_degree}")
print(f"Best test R² score: {max(test_scores):.4f}")

---

## 2.4 Regularization: Ridge, Lasso, ElasticNet

### Problem: Overfitting
When model is too complex, it fits training data perfectly but fails on new data.

### Solution: Regularization
Add a penalty for large coefficients to prevent overfitting.

| Method | Penalty | Effect |
|--------|---------|--------|
| **Ridge (L2)** | $\lambda \sum b_i^2$ | Shrinks coefficients (never zero) |
| **Lasso (L1)** | $\lambda \sum |b_i|$ | Can make coefficients zero (feature selection) |
| **ElasticNet** | Both L1 + L2 | Combines benefits of both |

In [ ]:
# Generate data with many features (some irrelevant)
from sklearn.datasets import make_regression

X, y = make_regression(n_samples=100, n_features=20, n_informative=5, 
                       noise=10, random_state=42)

# Add feature names
feature_names = [f'Feature_{i}' for i in range(20)]
X = pd.DataFrame(X, columns=feature_names)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Training: {len(X_train)} | Testing: {len(X_test)}")

In [ ]:
# Compare Regularization Methods

models = {
    'Linear Regression': LinearRegression(),
    'Ridge (L2)': Ridge(alpha=1.0),
    'Lasso (L1)': Lasso(alpha=1.0),
    'ElasticNet': ElasticNet(alpha=1.0, l1_ratio=0.5)
}

results = []

print("=== Model Comparison ===")
print(f"{'Model':<25} {'Train R²':>12} {'Test R²':>12} {'Non-zero coef':>15}")
print("-" * 65)

for name, model in models.items():
    model.fit(X_train, y_train)
    
    train_score = r2_score(y_train, model.predict(X_train))
    test_score = r2_score(y_test, model.predict(X_test))
    non_zero = np.sum(model.coef_ != 0)
    
    results.append({
        'Model': name,
        'Train R²': train_score,
        'Test R²': test_score,
        'Non-zero': non_zero
    })
    
    print(f"{name:<25} {train_score:>12.4f} {test_score:>12.4f} {non_zero:>15}")

print("\n✓ Lasso reduces features by setting coefficients to zero!")

In [ ]:
# Visualize Coefficient Comparison

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (name, model) in zip(axes.ravel(), models.items()):
    coefs = model.coef_
    colors = ['green' if c > 0 else 'red' for c in coefs]
    
    ax.barh(range(len(coefs)), coefs, color=colors, alpha=0.7)
    ax.set_yticks(range(len(coefs)))
    ax.set_yticklabels(feature_names, fontsize=8)
    ax.set_xlabel('Coefficient Value')
    ax.set_title(f'{name}\n(Non-zero: {np.sum(coefs != 0)}/20)', fontsize=11, fontweight='bold')
    ax.axvline(x=0, color='black', linewidth=0.5)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observations:")
print("- Linear Regression: All features have non-zero coefficients")
print("- Ridge: All features used, but shrunk")
print("- Lasso: Only important features kept (others = 0)")
print("- ElasticNet: Balanced approach")

In [ ]:
# Effect of Alpha (Regularization Strength)

alphas = [0.001, 0.01, 0.1, 1, 10, 100]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ridge
ridge_test_scores = []
for alpha in alphas:
    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train)
    ridge_test_scores.append(r2_score(y_test, model.predict(X_test)))

axes[0].plot(alphas, ridge_test_scores, 'bo-', linewidth=2, markersize=8)
axes[0].set_xscale('log')
axes[0].set_xlabel('Alpha (log scale)', fontsize=12)
axes[0].set_ylabel('Test R²', fontsize=12)
axes[0].set_title('Ridge: Effect of Alpha', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Lasso
lasso_test_scores = []
lasso_nonzero = []
for alpha in alphas:
    model = Lasso(alpha=alpha, max_iter=10000)
    model.fit(X_train, y_train)
    lasso_test_scores.append(r2_score(y_test, model.predict(X_test)))
    lasso_nonzero.append(np.sum(model.coef_ != 0))

ax2 = axes[1]
ax2.plot(alphas, lasso_test_scores, 'ro-', linewidth=2, markersize=8, label='Test R²')
ax2.set_xscale('log')
ax2.set_xlabel('Alpha (log scale)', fontsize=12)
ax2.set_ylabel('Test R²', fontsize=12)
ax2.set_title('Lasso: Effect of Alpha', fontsize=13, fontweight='bold')

# Add non-zero count on secondary axis
ax2b = ax2.twinx()
ax2b.plot(alphas, lasso_nonzero, 'g--', linewidth=2, markersize=8, label='Non-zero features')
ax2b.set_ylabel('Non-zero Features', fontsize=12, color='green')
ax2b.tick_params(axis='y', labelcolor='green')

ax2.legend(loc='upper left')
ax2b.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observations:")
print("- Small alpha: Less regularization, more complex model")
print("- Large alpha: More regularization, simpler model")
print("- For Lasso: Higher alpha = fewer features selected")

---

## 2.5 Regression Metrics Summary

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **MAE** | Mean Absolute Error | Average error in same units |
| **MSE** | Mean Squared Error | Penalizes large errors |
| **RMSE** | √MSE | Same units as target |
| **R²** | 1 - (SS_res/SS_tot) | % variance explained (0-1) |

In [ ]:
# Regression Metrics Demonstration

# Sample predictions
y_true = np.array([100, 150, 200, 250, 300, 350, 400])
y_pred = np.array([110, 145, 210, 240, 295, 360, 380])

# Calculate all metrics
mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_true, y_pred)

print("=== Regression Metrics ===")
print(f"\nMean Absolute Error (MAE): {mae:.2f}")
print(f"  → Average prediction is off by ${mae:.2f}")
print(f"\nMean Squared Error (MSE): {mse:.2f}")
print(f"  → Squared error (penalizes large errors)")
print(f"\nRoot Mean Squared Error (RMSE): {rmse:.2f}")
print(f"  → Same units as target, more interpretable than MSE")
print(f"\nR² Score: {r2:.4f}")
print(f"  → Model explains {r2*100:.1f}% of variance in data")

# Visualize
plt.figure(figsize=(10, 6))
x_pos = range(len(y_true))
width = 0.35

plt.bar([x - width/2 for x in x_pos], y_true, width, label='Actual', alpha=0.8)
plt.bar([x + width/2 for x in x_pos], y_pred, width, label='Predicted', alpha=0.8)

# Add error lines
for i, (actual, pred) in enumerate(zip(y_true, y_pred)):
    plt.plot([i, i], [actual, pred], 'r-', linewidth=2)
    error = abs(actual - pred)
    plt.annotate(f'{error}', xy=(i, (actual+pred)/2), fontsize=9, color='red')

plt.xlabel('Sample', fontsize=12)
plt.ylabel('Value', fontsize=12)
plt.title(f'Actual vs Predicted\nMAE: {mae:.2f} | RMSE: {rmse:.2f} | R²: {r2:.3f}', 
          fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---

## 📝 Practice Exercises

### Exercise 1: Simple Linear Regression

In [ ]:
# Exercise 1: Predict student scores based on study hours

# Data
study_hours = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]).reshape(-1, 1)
exam_scores = np.array([45, 50, 58, 62, 68, 72, 78, 84, 89, 95])

# TODO: Complete the following
# 1. Create and train a LinearRegression model
# 2. Calculate R² score
# 3. Predict score for 12 hours of study
# 4. Plot the data and regression line

# YOUR CODE HERE:
# model = ...
# model.fit(...)
# r2 = ...
# prediction_12_hours = ...

print("Complete the TODO items above!")

In [ ]:
# Solution to Exercise 1 (run after attempting)

model = LinearRegression()
model.fit(study_hours, exam_scores)

r2 = r2_score(exam_scores, model.predict(study_hours))
prediction_12_hours = model.predict([[12]])[0]

print("=== Exercise 1 Solution ===")
print(f"Slope: {model.coef_[0]:.2f} points per hour")
print(f"Intercept: {model.intercept_:.2f}")
print(f"R² Score: {r2:.4f}")
print(f"Predicted score for 12 hours: {prediction_12_hours:.1f}")

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(study_hours, exam_scores, s=100, alpha=0.6, label='Actual')
plt.plot(study_hours, model.predict(study_hours), 'r-', linewidth=2, label='Model')
plt.scatter([[12]], [prediction_12_hours], s=150, c='green', marker='*', 
            label=f'Prediction (12h): {prediction_12_hours:.1f}', zorder=5)
plt.xlabel('Study Hours')
plt.ylabel('Exam Score')
plt.title(f'Study Hours vs Exam Scores\nR² = {r2:.4f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Exercise 2: Compare Regularization

In [ ]:
# Exercise 2: Compare Linear, Ridge, and Lasso on the house price data

# Use the house price data from earlier
data = {
    'Size': [1400, 1600, 1700, 1875, 1100, 1550, 2350, 2450, 1425, 1700,
             1800, 2000, 2200, 1650, 1900, 2100, 2300, 1750, 2050, 1950],
    'Bedrooms': [3, 3, 2, 4, 2, 3, 4, 4, 3, 3, 3, 4, 4, 3, 3, 4, 5, 3, 4, 3],
    'Age': [0, 10, 15, 2, 20, 8, 5, 3, 12, 7, 6, 4, 1, 9, 11, 5, 2, 8, 3, 6],
    'Price': [245000, 312000, 279000, 308000, 199000, 219000, 405000, 424000,
              319000, 255000, 285000, 330000, 395000, 275000, 295000, 365000,
              440000, 290000, 355000, 320000]
}
df = pd.DataFrame(data)

X = df[['Size', 'Bedrooms', 'Age']]
y = df['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TODO: 
# 1. Train LinearRegression, Ridge, and Lasso models
# 2. Calculate R² for each on test set
# 3. Print the coefficients for each
# 4. Which model performs best? Why?

# YOUR CODE HERE:
print("Complete the exercise!")

---

## ✅ Phase Completion Checklist

- [ ] Understand Simple Linear Regression (y = mx + b)
- [ ] Build Multiple Linear Regression with several features
- [ ] Apply Polynomial Regression for non-linear data
- [ ] Use Ridge (L2) regularization to prevent overfitting
- [ ] Use Lasso (L1) for feature selection
- [ ] Understand ElasticNet as a combination
- [ ] Calculate and interpret MAE, MSE, RMSE, R²
- [ ] Complete all practice exercises

---

## 🎯 Key Takeaways

1. **Linear Regression** is the foundation - understand it well!
2. **Multiple features** often improve predictions
3. **Polynomial Regression** handles curved relationships
4. **Regularization** prevents overfitting:
   - Ridge: Shrinks all coefficients
   - Lasso: Can eliminate features (coefficients = 0)
   - ElasticNet: Best of both worlds
5. **Evaluation metrics** help choose the best model

---

## 📚 Next Steps

👉 **[Phase-2-Classification.ipynb](Phase-2-Classification.ipynb)** - Learn to predict categories!

Classification is equally important as regression. You'll learn:
- Logistic Regression
- K-Nearest Neighbors
- Decision Trees & Random Forest
- Support Vector Machines

**Happy Learning! 🚀**